# 08_Preproduccion — Pipeline completo y modelo final

Integra (análisis estático de A_01..A_07): carga del CSV original → calidad → features finalistas → modelo final.
No se serializan pipelines ni modelos (responsabilidad de A_09).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

csv_path = PROJECT_ROOT / "02_datos" / "01_Originales" / "contratacion_fondos.csv"
print(f"CSV esperado en: {csv_path.resolve()}")

CSV esperado en: C:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Ciencia_Datos_2\Caso_Agentes\02_datos\01_Originales\contratacion_fondos.csv


In [2]:
df = pd.read_csv(csv_path, index_col=0, encoding='utf-8')
print(df.shape)

(41188, 17)


In [3]:
from sklearn.model_selection import train_test_split

pk = df.index.to_numpy()
train_idx, val_idx = train_test_split(pk, test_size=0.30, random_state=42,
                                      stratify=df['contrata_fondos'])
df_train = df.loc[train_idx].copy()
df_val   = df.loc[val_idx].copy()
print(f"Train: {df_train.shape} | Validation: {df_val.shape}")

Train: (28831, 17) | Validation: (12357, 17)


In [4]:
import re

def normalizar_nombre(col):
    nombre = col.lower()
    for acento, sin_acento in [('á', 'a'), ('é', 'e'), ('í', 'i'),
                               ('ó', 'o'), ('ú', 'u'), ('ñ', 'n'), ('ü', 'u')]:
        nombre = nombre.replace(acento, sin_acento)
    nombre = re.sub(r'[^a-z0-9]+', '_', nombre)
    nombre = re.sub(r'_+', '_', nombre).strip('_')
    return nombre

df = df_train.copy()
propuesta = pd.DataFrame({
    'nombre_actual': df.columns,
    'nombre_normalizado': [normalizar_nombre(c) for c in df.columns],
})
mapeo = dict(zip(propuesta['nombre_actual'], propuesta['nombre_normalizado']))
mapeo['Fomación'] = 'formacion'
df = df.rename(columns=mapeo)

In [5]:
CATEGORICAS = ['trabajo', 'estado_civil', 'formacion', 'impago',
               'prestamo_hipotecario', 'prestamo_personal',
               'canal_de_contacto', 'mes', 'dia_de_la_semana',
               'resultado_campana_anterior']
for col in CATEGORICAS:
    df[col] = df[col].astype('category')

In [6]:
df = df.drop_duplicates(keep='first').reset_index(drop=True)
print(df.shape)

(28015, 17)


In [7]:
df = df.drop(columns=['dia_de_la_semana'])
df['formacion'] = df['formacion'].fillna('unknown')
mediana_edad = df['edad'].median()
df['edad'] = df['edad'].fillna(mediana_edad)

In [8]:
df['edad'] = df['edad'].astype('int64')

In [9]:
from sklearn.preprocessing import (OrdinalEncoder, OneHotEncoder,
                                   StandardScaler, FunctionTransformer)
from sklearn.impute import SimpleImputer

TARGET = 'contrata_fondos'
MAP_MES = {'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
           'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12}
cols_fase1_numericas = []
cols_fase2_noescalar = []
cols_intermedias_excluir = []

In [10]:
df_f1 = pd.DataFrame(index=df.index)
ORDEN_FORMACION = ['illiterate', 'basic.4y', 'basic.6y', 'basic.9y',
                   'high.school', 'professional.course', 'university.degree']

oe_form = OrdinalEncoder(categories=[ORDEN_FORMACION],
                         handle_unknown='use_encoded_value', unknown_value=np.nan)
df_f1['formacion_oe'] = oe_form.fit_transform(df[['formacion']])[:, 0]
imp_form = SimpleImputer(strategy='median')
df_f1['formacion_oe_imp'] = imp_form.fit_transform(df_f1[['formacion_oe']])[:, 0]
cols_fase1_numericas.append('formacion_oe_imp')
cols_intermedias_excluir += ['formacion', 'formacion_oe']

df_f1['num_dias_rec'] = df['num_dias_ultimo_contacto'].replace(-1, np.nan)
imp_dias = SimpleImputer(strategy='median')
df_f1['num_dias_ultimo_contacto_imp'] = imp_dias.fit_transform(df_f1[['num_dias_rec']])[:, 0]
cols_fase1_numericas.append('num_dias_ultimo_contacto_imp')
cols_intermedias_excluir += ['num_dias_ultimo_contacto', 'num_dias_rec']

log1p_t = FunctionTransformer(func=np.log1p, validate=False)
df_f1['num_contactos_esta_campana_log'] = log1p_t.fit_transform(df[['num_contactos_esta_campana']]).to_numpy()[:, 0]
df_f1['num_contactos_otras_campanas_log'] = log1p_t.fit_transform(df[['num_contactos_otras_campanas']]).to_numpy()[:, 0]
cols_fase1_numericas += ['num_contactos_esta_campana_log', 'num_contactos_otras_campanas_log']
cols_intermedias_excluir += ['num_contactos_esta_campana', 'num_contactos_otras_campanas']

for c in ['edad', 'variacion_tasa_empleo', 'euribor3m']:
    df_f1[c] = df[c].astype(float)
cols_fase1_numericas += ['edad', 'variacion_tasa_empleo', 'euribor3m']
cols_intermedias_excluir += ['edad', 'variacion_tasa_empleo', 'euribor3m']

In [11]:
df_f2 = pd.DataFrame(index=df.index)
CATS_OHE = ['trabajo', 'estado_civil', 'impago', 'prestamo_hipotecario',
            'prestamo_personal', 'canal_de_contacto', 'resultado_campana_anterior']
for col in CATS_OHE:
    ohe = OneHotEncoder(drop='first', sparse_output=False, dtype=np.int8)
    arr = ohe.fit_transform(df[[col]])
    nombres = [f'{col}_{cat}' for cat in ohe.categories_[0][1:]]
    for i, nombre in enumerate(nombres):
        df_f2[nombre] = arr[:, i]
    cols_intermedias_excluir.append(col)
cols_fase2_noescalar.extend(df_f2.columns)

df_f2['contactado_previamente'] = (df['num_dias_ultimo_contacto'] >= 0).astype(np.int8)
cols_fase2_noescalar.append('contactado_previamente')

mes_num = df['mes'].map(MAP_MES).astype(float)
angulo = 2 * np.pi * mes_num / 12
df_f2['mes_sin'] = np.sin(angulo)
df_f2['mes_cos'] = np.cos(angulo)
cols_fase2_noescalar += ['mes_sin', 'mes_cos']
cols_intermedias_excluir.append('mes')

In [12]:
ss = StandardScaler()
arr_ss = ss.fit_transform(df_f1[cols_fase1_numericas])
df_f3 = pd.DataFrame(arr_ss,
                     columns=[f'{c}_ss' for c in cols_fase1_numericas],
                     index=df.index)
cols_intermedias_excluir += cols_fase1_numericas
cols_finales_escaladas = list(df_f3.columns)

In [13]:
df_final = pd.concat([df[[TARGET]], df_f2, df_f3], axis=1)
df = df_final.copy()
print(f"df_final: {df.shape}")

df_final: (28015, 34)


In [14]:
VARIABLES_FINALISTAS = [
    'trabajo_student', 'impago_unknown', 'prestamo_hipotecario_yes',
    'canal_de_contacto_telephone', 'mes_sin', 'mes_cos',
    'resultado_campana_anterior_nonexistent', 'resultado_campana_anterior_success',
    'edad_ss', 'num_contactos_esta_campana_log_ss',
    'variacion_tasa_empleo_ss', 'euribor3m_ss',
]
df_modelo = df_final[VARIABLES_FINALISTAS + [TARGET]].copy()
print(df_modelo.shape)

(28015, 13)


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

X = df_modelo.drop(columns=[TARGET])
y = df_modelo[TARGET].astype(int)

modelo_final = RandomForestClassifier(
    n_estimators=300, min_samples_split=2, min_samples_leaf=1, max_depth=10,
    max_features='sqrt', random_state=42, n_jobs=-1, class_weight=None,
)
modelo_final.fit(X, y)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
auc_cv = cross_val_score(modelo_final, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"AUC CV (3-fold): {auc_cv.mean():.4f} ± {auc_cv.std():.4f}")

AUC CV (3-fold): 0.8006 ± 0.0024
